# Notebook 1: Data Exploration

This notebook explores the Eye Disease Classification dataset.

## Objectives:
- Load and visualize sample images
- Analyze class distribution
- Examine image statistics
- Understand data quality

In [ ]:
# Imports
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import pandas as pd

# Add parent directory to path
sys.path.append('..')

from src.config import get_config, CLASS_NAMES, CLASS_NAMES_DISPLAY, CLASS_COLORS
from src.data_loader import get_class_distribution, print_dataset_info
from src.preprocessing import get_image_statistics

%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

## 1. Configuration

In [ ]:
# Load configuration
config = get_config()

data_dir = config.get('dataset.raw_data_path', '../data/raw')
processed_dir = config.get('dataset.processed_data_path', '../data/processed')
classes = config.get('dataset.classes', CLASS_NAMES)

print(f"Data directory: {data_dir}")
print(f"Classes: {classes}")

## 2. Class Distribution

In [ ]:
# Check raw data distribution
if os.path.exists(data_dir):
    print_dataset_info(data_dir, classes)
else:
    print(f"Raw data directory not found: {data_dir}")

In [ ]:
# Visualize class distribution
distribution = get_class_distribution(data_dir, classes)

fig, ax = plt.subplots(figsize=(10, 6))

class_names_display = [CLASS_NAMES_DISPLAY.get(c, c) for c in classes]
colors = [CLASS_COLORS.get(c, '#3498db') for c in classes]

bars = ax.bar(class_names_display, distribution.values(), color=colors)
ax.set_xlabel('Disease Class')
ax.set_ylabel('Number of Images')
ax.set_title('Dataset Class Distribution')
ax.tick_params(axis='x', rotation=45)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height)}',
            ha='center', va='bottom')

plt.tight_layout()
plt.show()

## 3. Sample Images

In [ ]:
# Display sample images from each class
def display_sample_images(data_dir, classes, samples_per_class=3):
    """Display sample images from each class."""
    import glob
    
    fig, axes = plt.subplots(len(classes), samples_per_class,
                            figsize=(15, 4*len(classes)))
    
    for i, cls in enumerate(classes):
        cls_dir = os.path.join(data_dir, cls)
        
        if not os.path.exists(cls_dir):
            continue
        
        # Get sample images
        images = glob.glob(os.path.join(cls_dir, '*.*'))[:samples_per_class]
        
        for j, img_path in enumerate(images):
            try:
                img = Image.open(img_path)
                
                if len(classes) == 1:
                    ax = axes[j]
                else:
                    ax = axes[i, j]
                
                ax.imshow(img)
                ax.axis('off')
                
                if j == 0:
                    display_name = CLASS_NAMES_DISPLAY.get(cls, cls)
                    ax.set_ylabel(display_name, fontsize=12, fontweight='bold')
            
            except Exception as e:
                print(f"Error loading {img_path}: {e}")
    
    plt.suptitle('Sample Images from Each Class', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

if os.path.exists(data_dir):
    display_sample_images(data_dir, classes)

## 4. Image Statistics

In [ ]:
# Analyze image statistics
if os.path.exists(data_dir):
    stats = get_image_statistics(data_dir)
    
    print("\nImage Statistics:")
    print(f"Total images analyzed: {stats['total_images']}")
    print(f"Corrupt images: {stats['corrupt_images']}")
    print(f"\nDimensions:")
    print(f"  Average: {stats['avg_width']:.1f} x {stats['avg_height']:.1f}")
    print(f"  Min: {stats['min_width']} x {stats['min_height']}")
    print(f"  Max: {stats['max_width']} x {stats['max_height']}")

## 5. Data Quality Check

In [ ]:
# Check for corrupt images
from src.preprocessing import verify_image
import glob

print("Checking for corrupt images...")
corrupt_images = []

for cls in classes:
    cls_dir = os.path.join(data_dir, cls)
    if os.path.exists(cls_dir):
        for img_path in glob.glob(os.path.join(cls_dir, '*.*')):
            if not verify_image(img_path):
                corrupt_images.append(img_path)

if corrupt_images:
    print(f"\nFound {len(corrupt_images)} corrupt images:")
    for img in corrupt_images[:10]:
        print(f"  - {img}")
    if len(corrupt_images) > 10:
        print(f"  ... and {len(corrupt_images) - 10} more")
else:
    print("\n✅ No corrupt images found!")

## 6. Summary

This notebook provided an overview of the dataset including:
- Class distribution
- Sample images
- Image statistics
- Data quality check

Next steps:
1. Run preprocessing: `python main.py preprocess`
2. Train models: `python main.py train --all`